# Rüzgar Colab Kurulumu

Bu defter:
- `umithaymana/ruzgar` deposunu alır (**önce git clone**, olmazsa **GitHub ZIP**)
- uygun `requirements` dosyalarını kurar
- GPU durumunu kontrol eder
- `main.py` çalıştırma komutunu hazırlar

In [ ]:
# 1) Depoyu indir (git veya ZIP yedeği)
# Colab'da bazen `git clone` başarısız olur (git yok, ağ, GitHub kısıtı).
import os, shutil, subprocess, urllib.request, zipfile

REPO_GIT = "https://github.com/umithaymana/ruzgar.git"
REPO_DIR = "/content/ruzgar"
ZIP_URLS = [
    "https://github.com/umithaymana/ruzgar/archive/refs/heads/main.zip",
    "https://github.com/umithaymana/ruzgar/archive/refs/heads/master.zip",
]

def ensure_git():
    from shutil import which
    if which("git"):
        return
    print("Git kuruluyor...")
    subprocess.run(
        ["apt-get", "update", "-qq"],
        check=False,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
    subprocess.run(
        ["apt-get", "install", "-y", "-qq", "git"],
        check=True,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.PIPE,
        text=True,
    )

def clone_repo():
    ensure_git()
    if os.path.exists(REPO_DIR):
        print(f"Temizleniyor: {REPO_DIR}")
        shutil.rmtree(REPO_DIR)
    print("Git ile klonlanıyor (sığ kopya)...")
    r = subprocess.run(
        ["git", "clone", "--depth", "1", REPO_GIT, REPO_DIR],
        capture_output=True,
        text=True,
    )
    if r.returncode != 0:
        print("GIT STDERR:\n", r.stderr)
        print("GIT STDOUT:\n", r.stdout)
        raise RuntimeError("git clone başarısız.")
    print("Git ile tamam.")

def download_zip_fallback():
    if os.path.exists(REPO_DIR):
        shutil.rmtree(REPO_DIR)
    zip_path = "/content/_ruzgar_repo.zip"
    last_err = None
    for zurl in ZIP_URLS:
        try:
            print("ZIP deneniyor:", zurl)
            urllib.request.urlretrieve(zurl, zip_path)
            break
        except Exception as e:
            last_err = e
            print("ZIP indirilemedi:", e)
    else:
        raise RuntimeError(f"ZIP indirilemedi: {last_err}")

    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall("/content")
    os.remove(zip_path)

    extracted = "/content/ruzgar-main"
    if not os.path.isdir(extracted):
        extracted = "/content/ruzgar-master"
    if not os.path.isdir(extracted):
        raise RuntimeError("ZIP açıldı ama klasör adı beklenen değil (main/master).")
    shutil.move(extracted, REPO_DIR)
    print("ZIP ile tamam.")

try:
    clone_repo()
except Exception as e:
    print("Git başarısız, ZIP ile deneniyor...", e)
    download_zip_fallback()

os.chdir(REPO_DIR)
print("Çalışma dizini:", os.getcwd())


In [ ]:
# 2) Gerekli kütüphaneleri kur
# UYARI: ilim-voice/requirements.txt içinde TTS (Coqui) vardır — Colab'da ASLA onu kullanmayın.
# Sadece requirements-colab.txt; yokses modülü atlanır.
# Tam kurulum hâlâ hata verirse ortam değişkeni: os.environ["COLAB_SKIP_ILIM_VOICE"] = "1"
import os, subprocess

def pip_install_line_by_line(req_path, label=""):
    """Bir satır hata verse bile diğerlerine devam eder; hangi paket kırıldı görünür."""
    failed = []
    with open(req_path, encoding="utf-8") as f:
        lines = [ln.strip() for ln in f if ln.strip() and not ln.strip().startswith("#")]
    for spec in lines:
        print(f"  → {spec}")
        r = subprocess.run(
            ["python", "-m", "pip", "install", "-q", spec],
            capture_output=True,
            text=True,
        )
        if r.returncode != 0:
            print(f"  !! Hata ({label}): {spec}")
            print((r.stderr or r.stdout or "")[-800:])
            failed.append(spec)
    return failed

def build_req_list():
    out = [
        "ilim-assistant/requirements.txt",
        "ilim-video/requirements.txt",
    ]
    vcol = "ilim-voice/requirements-colab.txt"
    if os.environ.get("COLAB_SKIP_ILIM_VOICE", "").strip() in ("1", "true", "yes"):
        print("COLAB_SKIP_ILIM_VOICE: ilim-voice atlandı.")
    elif os.path.exists(vcol):
        out.append(vcol)
    else:
        print("UYARI: ilim-voice/requirements-colab.txt yok — ses modülü atlanıyor (TTS'li requirements.txt dokunulmadı).")
    return [p for p in out if os.path.exists(p)]

reqs = build_req_list()
if not reqs:
    raise FileNotFoundError("requirements dosyası bulunamadı.")

subprocess.check_call(["python", "-m", "pip", "install", "--upgrade", "pip"])

for req in reqs:
    print(f"\nKuruluyor: {req}")
    if "ilim-voice" in req.replace("\\", "/"):
        failed = pip_install_line_by_line(req, label="ilim-voice")
        if failed:
            print("\nÖzet — kurulamayan satırlar:", failed)
            print("İsterseniz ses modülünü tamamen kapatın: os.environ['COLAB_SKIP_ILIM_VOICE']='1' sonra bu hücreyi yeniden çalıştırın.")
    else:
        subprocess.check_call(["python", "-m", "pip", "install", "-r", req])

print("\nKurulum tamam.")

In [ ]:
# 3) GPU kontrolü
import subprocess, torch

print("CUDA kullanılabilir mi?:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU görünmüyor. Colab Runtime > Change runtime type > GPU seçin.")

print("\nNVIDIA-SMI çıktısı:")
try:
    print(subprocess.check_output(["nvidia-smi"], text=True))
except Exception as e:
    print("nvidia-smi çalıştırılamadı:", e)

In [ ]:
# 4) main.py başlatmaya hazır hale getir
import os

MAIN_PATH = "main.py"
ALT_PATHS = [
    "ilim-assistant/desktop_server.py",
    "ilim-assistant/gradio_chat.py",
]

if os.path.exists(MAIN_PATH):
    RUN_CMD = f"python {MAIN_PATH}"
    print("Hazır komut:", RUN_CMD)
else:
    print("main.py bulunamadı.")
    found = [p for p in ALT_PATHS if os.path.exists(p)]
    if found:
        print("Projede bulunan çalıştırılabilir alternatifler:")
        for p in found:
            print(" -", p)
        print("\nÖrnek:")
        print(" !python ilim-assistant/desktop_server.py")
    else:
        print("Çalıştırılabilir ana giriş dosyası bulunamadı. Repo yapısını kontrol edin.")